# Hafta 9 — Ek (b): Hafta 8 GPT'yi BPE tokenizer ile eğit (Colab T4)


Hafta 8 ile tek fark tokenizer: karakter (vocab 104) yerine `RegexTokenizer` + `TR_SPLIT_PATTERN` (vocab 512). Model ve hiperparametreler aynı → hafta 8 sonucu: **val 1.2454 (karakter başına), 701 sn**.

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


## 1. Dosyaları yükle
Hepsini birlikte seç:
- `week08/`: `layers.py`, `dataset.py`, `turkce.txt`
- `week09/`: `bpe.py`, `regex_tokenizer.py`

Lokalde çalıştırıyorsan bu hücreyi atla.

In [4]:
from google.colab import files

uploaded = files.upload()
!ls -la

Saving bpe.py to bpe.py
Saving regex_tokenizer.py to regex_tokenizer.py
total 1220
drwxr-xr-x 1 root root    4096 Oct 15 15:42 .
drwxr-xr-x 1 root root    4096 Oct 15 15:34 ..
-rw-r--r-- 1 root root    2560 Oct 15 15:42 bpe.py
drwxr-xr-x 4 root root    4096 Sep 30 13:28 .config
-rw-r--r-- 1 root root    1713 Oct 15 15:41 dataset.py
-rw-r--r-- 1 root root    6941 Oct 15 15:41 layers.py
-rw-r--r-- 1 root root    3134 Oct 15 15:42 regex_tokenizer.py
drwxr-xr-x 1 root root    4096 Sep 30 13:28 sample_data
-rw-r--r-- 1 root root 1208996 Oct 15 15:41 turkce.txt


## 2. Import'lar

In [5]:
import os
import sys
import time
import json
sys.path.append("../week08")   # lokalde week08/ modülleri; Colab'da dosyalar zaten aynı klasörde
import torch
from layers import GPTLanguageModel
from dataset import read_text, Dataset
from regex_tokenizer import RegexTokenizer, TR_SPLIT_PATTERN

In [13]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## 3. Hiperparametreler (hafta 8 `video` config ile birebir aynı) + device

In [15]:
N_EMBD, N_HEAD, N_LAYER = 384, 6, 6
BLOCK_SIZE, BATCH_SIZE = 256, 64
DROPOUT = 0.2
LR = 3e-4
MAX_STEPS, EVAL_INTERVAL = 2000, 250
EVAL_ITERS = 200
SEED = 1337
VOCAB_SIZE = 512

if torch.cuda.is_available():
    DEVICE = "cuda"   # Colab T4
elif torch.backends.mps.is_available():
    DEVICE = "mps"    # Apple M1
else:
    DEVICE = "cpu"
print(DEVICE)

cuda


## 4. Tokenizer eğitimi + veriyi encode et
Tokenizer eğitimi gradient'siz, sadece sayma (~1 dk). Sonuç `bpe_cache.pt`'ye kaydedilir; ikinci çalıştırmada oradan yüklenir.

In [7]:
path = "turkce.txt" if os.path.exists("turkce.txt") else "../week08/turkce.txt"
text = read_text(path)

CACHE = "bpe_cache.pt"
tok = RegexTokenizer(TR_SPLIT_PATTERN)
if os.path.exists(CACHE):
    cache = torch.load(CACHE)
    tok.merges = cache["merges"]
    tok._build_vocab()
    data = cache["data"]
else:
    t0 = time.time()
    tok.train(text, VOCAB_SIZE)
    data = torch.tensor(tok.encode(text), dtype=torch.long)
    torch.save({"merges": tok.merges, "data": data}, CACHE)
    print(f"tokenizer + encode: {time.time() - t0:.0f} sn")

print(f"karakter: {len(text):,}  token: {len(data):,}  max id: {data.max().item()}")

tokenizer + encode: 113 sn
karakter: 1,101,042  token: 573,620  max id: 511


## 5. Dataset + model

In [8]:
torch.manual_seed(SEED)
ds = Dataset(data, BLOCK_SIZE, BATCH_SIZE, device=DEVICE)
model = GPTLanguageModel(VOCAB_SIZE, N_EMBD, BLOCK_SIZE, n_head=N_HEAD, n_layer=N_LAYER,
                         use_ffwd=True, use_residual=True, use_ln=True, dropout=DROPOUT)
model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"{DEVICE} | vocab: {VOCAB_SIZE} | {n_params:,} parametre")

cuda | vocab: 512 | 11,132,672 parametre


In [9]:
@torch.no_grad()
def estimate_loss(model, ds, eval_iters):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = ds.get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


## 6. Eğitim (T4'te hafta 8'de ~12 dk sürmüştü)

In [16]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
t0 = time.time()
best_val = float("inf")
os.makedirs("checkpoints", exist_ok=True)
for step in range(MAX_STEPS):
    if step % EVAL_INTERVAL == 0:
        out = estimate_loss(model, ds, EVAL_ITERS)
        print(f"step: {step}, train_loss: {out['train']:.4f}, val_loss: {out['val']:.4f}")
        if out['val'] < best_val:
              best_val = out['val']
              torch.save({'model': model.state_dict(), 'merges': tok.merges, 'step': step,
                          'val': best_val}, "checkpoints/turkce_bpe512_best.pt")

    xb, yb = ds.get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

elapsed = time.time() - t0
out = estimate_loss(model, ds, EVAL_ITERS)
print(f"train_loss: {out['train']:.4f}, val_loss: {out['val']:.4f}, süre: {elapsed:.0f} sn")

step: 0, train_loss: 3.3978, val_loss: 3.5698
step: 250, train_loss: 2.6949, val_loss: 2.9949
step: 500, train_loss: 2.2634, val_loss: 2.6493
step: 750, train_loss: 2.0381, val_loss: 2.5093
step: 1000, train_loss: 1.8782, val_loss: 2.4561
step: 1250, train_loss: 1.7443, val_loss: 2.4137
step: 1500, train_loss: 1.6316, val_loss: 2.4101
step: 1750, train_loss: 1.5160, val_loss: 2.4213
train_loss: 1.4075, val_loss: 2.4372, süre: 485 sn


## 7. Hafta 8 ile karşılaştırma: karakter başına loss
Loss token başına; bir token ~1.9 karakter. Toplam loss aynı kalsın diye token başına loss'u token başına düşen karakter sayısına böleriz → hafta 8'in karakter başına loss'uyla aynı birim (nats/karakter).

In [17]:
WEEK8_VAL = 1.2454   # hafta 8, karakter tokenizer, aynı model

val_chars = len(tok.decode(ds.val_data.tolist()))
chars_per_token = val_chars / len(ds.val_data)
val_per_char = out['val'] / chars_per_token
train_per_char = out['train'] / chars_per_token

print(f"val: {len(ds.val_data):,} token, {val_chars:,} karakter → {chars_per_token:.2f} karakter/token")
print(f"token başına     val: {out['val']:.4f}")
print(f"karakter başına  val: {val_per_char:.4f}  (hafta 8: {WEEK8_VAL})")

val: 57,362 token, 110,827 karakter → 1.93 karakter/token
token başına     val: 2.4372
karakter başına  val: 1.2614  (hafta 8: 1.2454)


## 8. Örnek metin (500 token ≈ hafta 8'deki 1000 karakter)

In [18]:
model.eval()
start = torch.tensor([tok.encode("\n")], device=DEVICE)
sample = tok.decode(model.generate(start, max_new_tokens=500)[0].tolist())
print(sample)


Mustafa Kemal

Efendiler, vaziyeti haberdâr olup Cemal Paşa’dan bazı maksadını tahtında da takarrürüne tebaiyet etmiştim, Cumhuriyet’in ilânına tevfîkanın İsmet Paşa’ya yazıldığı diğer mektup şu cümle hulâsa etmiştim: “4 Miralay Kâzım Paşa, bu prensipler verilen evim adamlarla temas ve tatbike çalışanları ufak bir mahallerde, Hamdi Paşa, Müfit Bey, diğer bir şey olmadığını tehdit etmek diyorlar. Celâlettin Arif Bey, Gelibolu’da, Hüseyin Kâzım Bey’e yazdığı tarihiyle bana da yazdım. Süleyman Bey, telefon bunu bildirdi. Fakat şu cevâbı verdi:

29.336

Harbiye Nâzırı Cemal Paşa Hazretlerine

Hüseyin

Cemal

Efendiler, Garbî Trabzon’da Bekir Sami Bey’in verdiği cevâben, şu idi:

Şifre

Bekir Sami Bey
Beşikten sonra faide kürsüden ise sözleri, sarf-ı nazar ederek, şâmil ve riayet edebilir miydi? Bize, bu sözlerin şahına kadar cür’et bir şifre telgraf yazıldı. İstîzâhımızdan gelemezse derecekleri istediler. Kendisine, Ethem Efendiler, Reşit Bey’e, aynı tarihli şifre tel


## 9. Kaydet + indir
Lokalde `week09/` içine aç.

In [19]:
NAME = "turkce_bpe512_video"
config = dict(n_embd=N_EMBD, n_head=N_HEAD, n_layer=N_LAYER, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE,
              dropout=DROPOUT, lr=LR, max_steps=MAX_STEPS, eval_interval=EVAL_INTERVAL)
results = dict(config=config, device=DEVICE, vocab_size=VOCAB_SIZE, params=n_params, time=elapsed,
               train=out['train'], val=out['val'], chars_per_token=chars_per_token,
               train_per_char=train_per_char, val_per_char=val_per_char, week8_val_per_char=WEEK8_VAL)

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("samples", exist_ok=True)
torch.save({'model': model.state_dict(), 'config': config, 'merges': tok.merges}, f"checkpoints/{NAME}.pt")
with open(f"samples/{NAME}.txt", "w", encoding="utf-8") as f:
    f.write(sample)
with open(f"samples/{NAME}.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

{
  "config": {
    "n_embd": 384,
    "n_head": 6,
    "n_layer": 6,
    "block_size": 256,
    "batch_size": 64,
    "dropout": 0.2,
    "lr": 0.0003,
    "max_steps": 2000,
    "eval_interval": 250
  },
  "device": "cuda",
  "vocab_size": 512,
  "params": 11132672,
  "time": 484.5099127292633,
  "train": 1.4074790477752686,
  "val": 2.437173843383789,
  "chars_per_token": 1.9320630382483177,
  "train_per_char": 0.7284850545307998,
  "val_per_char": 1.2614359858534554,
  "week8_val_per_char": 1.2454
}


In [20]:
!zip -r week9_colab_results.zip samples checkpoints
from google.colab import files
files.download('week9_colab_results.zip')

  adding: samples/ (stored 0%)
  adding: samples/turkce_bpe512_video.txt (deflated 41%)
  adding: samples/turkce_bpe512_video.json (deflated 45%)
  adding: checkpoints/ (stored 0%)
  adding: checkpoints/turkce_bpe512_best.pt (deflated 23%)
  adding: checkpoints/turkce_bpe512_video.pt (deflated 23%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>